# Part 3 · Semantic search over 300 reviews

**Building Agentic AI — Day 1, 11:30–12:15**

Welcome back. We previously computed cosine similarity between 8 sentences.
Now we do it *for real*: you've just joined **Playfield**, a small indie game
storefront, as its first data person. You have 20 games, 300 player reviews,
and one recurring request from the studios: *"what are players actually saying?"*

In this notebook you build the answer machine: **semantic search** — find reviews
by *meaning*, not keywords. It's the "R" in RAG, and on Day 3 your agent will
call the function you write today.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env", override=True)  # .env wins over stale keys inherited from the kernel's environment
client = genai.Client()

EMBED_MODEL = "gemini-embedding-001"

pd.set_option("display.max_colwidth", 100)

## 1. Meet the dataset

Two CSVs — the game catalog and the player reviews. (A crash course in `pandas`
happens naturally in the next ten cells; watch what each line does to the table.)

In [ ]:
games = pd.read_csv("../data/games.csv")
games

In [ ]:
reviews = pd.read_csv("../data/reviews.csv")
print(f"{len(reviews)} reviews")
reviews.head(3)

`pandas` one-liners answer the easy questions instantly:

In [ ]:
reviews["recommended"].mean()          # fraction of thumbs-up overall

In [ ]:
reviews["hours_played"].describe().round(1)

`merge` joins the two tables on `game_id`, so every review knows its game's
title, genre and price — we'll work with this combined table from now on:

In [ ]:
df = reviews.merge(games, on="game_id")
df[["review_id", "title", "recommended", "hours_played", "review_text"]].head(3)

Which games do players love — and which are in trouble?

In [ ]:
rating = df.groupby("title")["recommended"].mean().sort_values()
rating.plot(kind="barh", figsize=(7, 6), title="Share of positive reviews per game");

Read a few raw reviews to get a feel for the *texture* of this data — typos,
sarcasm, ALL CAPS, the occasional Romanian:

In [ ]:
for text in df.sample(3, random_state=7)["review_text"]:
    print("—" * 70)
    print(text)

## 2. Keyword search — and where it breaks

The obvious approach: `str.contains`.

In [ ]:
def keyword_search(query):
    hits = df[df["review_text"].str.contains(query, case=False, regex=False)]
    return hits[["title", "review_text"]]


keyword_search("crash").head()

Works! Until the player types something the reviewer didn't:

In [ ]:
print(len(keyword_search("crash")), "hits for 'crash'")
print(len(keyword_search("game keeps freezing")), "hits for 'game keeps freezing'")
print(len(keyword_search("predatory monetization")), "hits for 'predatory monetization'")

Zero hits — yet you *know* reviews complain about freezes and cash grabs; they just
use different words ("CTD", "hangs", "wallet warriors", "pay-to-win"...).
Keyword search matches **spelling**. We need to match **meaning**.

## 3. Embed the whole corpus (batched, cached, retry-safe)

Same as this morning, at scale: one 768-dim vector per review. Three bits of
engineering that mark the difference between a demo and something that survives
contact with reality:

- **batching** — the API takes up to ~100 texts per call, so we send chunks,
- **retries with backoff** — free-tier rate limits (HTTP 429) are a *when*, not an *if*,
- **caching** — embeddings don't change; compute once, save to disk, reload forever.

In [ ]:
CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)
EMB_FILE = CACHE / "review_embeddings.npy"


def embed_texts(texts, task_type, batch_size=64):
    """Embed a list of texts → normalized (len(texts), 768) matrix."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        for attempt in range(7):
            try:
                result = client.models.embed_content(
                    model=EMBED_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(
                        task_type=task_type,
                        output_dimensionality=768,
                    ),
                )
                vectors.extend(e.values for e in result.embeddings)
                break
            except genai.errors.APIError as err:
                wait = 2**attempt
                print(f"  API error {err.code}: {err.message}")
                print(f"  retrying in {wait}s…")
                time.sleep(wait)
        else:
            raise RuntimeError("embedding failed after 7 attempts")
        print(f"  embedded {min(start + batch_size, len(texts))}/{len(texts)}")
    E = np.array(vectors)
    return E / np.linalg.norm(E, axis=1, keepdims=True)


if EMB_FILE.exists():
    doc_vecs = np.load(EMB_FILE)
    print("loaded from cache ✅")
else:
    doc_vecs = embed_texts(df["review_text"].tolist(), task_type="RETRIEVAL_DOCUMENT")
    np.save(EMB_FILE, doc_vecs)

doc_vecs.shape

One subtlety: `task_type="RETRIEVAL_DOCUMENT"` for the corpus, but
`RETRIEVAL_QUERY` for searches. Queries and documents are phrased differently
("crashes on startup?" vs "the game died again, third time today") — Gemini
embeddings are trained to map the *question* and the *answer* close together
when you label which is which.

## 4. The search function — 15 lines, as promised

In [ ]:
def search(query, top_k=5):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=768,
        ),
    )
    q = np.array(result.embeddings[0].values)
    q = q / np.linalg.norm(q)

    scores = doc_vecs @ q                      # cosine similarity vs every review at once
    top = np.argsort(scores)[::-1][:top_k]     # best k indices

    hits = df.iloc[top][["review_id", "title", "recommended", "review_text"]].copy()
    hits.insert(0, "score", scores[top].round(3))
    return hits

That's the whole engine. Let's interrogate it — starting with the query that
keyword search fumbled:

In [ ]:
search("game keeps freezing")

In [ ]:
search("predatory monetization")

In [ ]:
search("this game made me cry")

In [ ]:
search("great to play with friends")

And because embeddings are multilingual — **search in Romanian, find English reviews**:

In [ ]:
search("jocul se blochează la încărcare")

Side-by-side, the difference in one picture:

In [ ]:
q = "game keeps freezing"
print(f"KEYWORD  '{q}' → {len(keyword_search(q))} hits")
print(f"SEMANTIC '{q}' → top hit: {search(q, top_k=1)['review_text'].iloc[0][:100]}…")

## 5. Exercises

**5.1 — Scoped search (⭐)** Write `search_game(query, title, top_k=5)` that only
searches within one game's reviews. Hint: build a boolean mask from `df["title"]`,
apply it to *both* `df` and `doc_vecs` (numpy accepts boolean masks too).

**5.2 — Most similar pair (⭐⭐)** Find the two *most similar reviews* in the whole
corpus. Hints: `S = doc_vecs @ doc_vecs.T`, kill the diagonal with
`np.fill_diagonal(S, -1)`, then `np.unravel_index(S.argmax(), S.shape)`.
Are they for the same game? Same complaint in different words?

**5.3 — More like this (⭐)** Write `more_like_this(review_id, top_k=5)`.
No new embedding needed — the review's vector is already a row of `doc_vecs`.

**5.4 — Honest search (⭐⭐)** Modify `search` to drop results scoring below a
threshold (try `0.5`). Then search for `"quantum physics homework"`.
Returning *nothing* is the correct answer — remember this on Day 3, when an
agent must decide whether its retrieval tool actually found anything.

In [ ]:
# 5.1 — your code here

In [ ]:
# 5.2 — your code here

In [ ]:
# 5.3 — your code here

In [ ]:
# 5.4 — your code here

---
## ✅ Checkpoint

You built a semantic search engine over a real(istic) corpus: batched + cached
embeddings, cosine ranking, honest no-answer handling. **You built the "R" in RAG.**

☕ **Short break.** In Part 4 the model starts working *for* you: turning messy
review prose into clean, analyzable rows — the last skill your Friday agent needs.